In [1]:
import pandas as pd
import numpy as np
from geopy.distance import EARTH_RADIUS


def haversin(theta):
    return (1 - np.cos(theta)) / 2.0

def haversine_2D_mat(data1, data2):
    lats1, lons1 = data1['lat'].values, data1['lon'].values
    phis1, lambs1 = np.radians(lats1).reshape(-1, 1), np.radians(lons1).reshape(-1, 1)

    lats2, lons2 = data2['lat'].values, data2['lon'].values
    phis2, lambs2 = np.radians(lats2).reshape(-1, 1), np.radians(lons2).reshape(-1, 1)

    deltas_lats = phis1 - phis2.T
    deltas_lons = lambs1 - lambs2.T

    cos_phis1 = np.cos(phis1)
    cos_phis2 = np.cos(phis2)
    a = haversin(deltas_lats) + cos_phis1 * cos_phis2.T * haversin(deltas_lons)

    vec_dist = 2 * EARTH_RADIUS * np.arcsin(np.sqrt(a))
    return vec_dist * 1000

In [2]:
df_base = pd.read_csv("data/moscow_geocoded.csv")#.dropna()
df_base = df_base[["lat", "lon", "rubrics"]].copy()

df_points = pd.read_excel("data/moscow_only.xlsx", nrows=1000)[["lat", "lng"]]
df_points = df_points.rename({"lng": "lon"}, axis=1)

rubrics = [
    ["Ресторан", "Кафе", "Место для пикника"],
    ["Суши-бар", "Доставка еды и обедов"],
    ["Автомойка", "Детейлинг"],
    ["Православный храм"],
]
R = 500

In [3]:
# uniq_rubricks = df_base['rubrics'].str.split(';').explode('rubric_list').value_counts()

In [4]:
df_base['uno_rubric'] = df_base['rubrics'].str.split(';')
exploded = df_base.explode('uno_rubric').dropna()
exploded

,lat,lon,rubrics,uno_rubric
3,55.625578,37.606392,Достопримечательность,Достопримечательность
4,55.625578,37.606392,Окна;Изготовление витражей;Реставрационная мас...,Окна
4,55.625578,37.606392,Окна;Изготовление витражей;Реставрационная мас...,Изготовление витражей
4,55.625578,37.606392,Окна;Изготовление витражей;Реставрационная мас...,Реставрационная мастерская
5,55.625578,37.606392,Магазин продуктов,Магазин продуктов
...,...,...,...,...
78762,55.969187,37.140172,Конный клуб,Конный клуб
78767,55.596424,37.927547,Место для пикника,Место для пикника
78773,55.764431,37.614644,Жанровая скульптура,Жанровая скульптура
78783,54.918647,39.179536,Место для пикника,Место для пикника


In [5]:
rubric_to_indices = exploded.groupby('uno_rubric').agg(
    {'lat': lambda x: df_base.index[x.index].tolist()}
).to_dict()['lat'] # Словарь 
rubric_to_indices

{'3D-услуги': [1583, 2996, 9700, 16596, 22742, 26650, 39439, 70168],
 'GPS-оборудование': [20934, 41913],
 'IP-телефония': [2272,
  10025,
  22269,
  30106,
  37136,
  58671,
  59945,
  61694,
  62229,
  63547,
  71398,
  72560,
  73036],
 'IT-компания': [724,
  1300,
  1378,
  3047,
  3575,
  4858,
  5379,
  5584,
  7336,
  9780,
  9916,
  9920,
  10484,
  11238,
  11898,
  13075,
  13883,
  14700,
  15264,
  15908,
  18533,
  20405,
  21049,
  23822,
  23893,
  24023,
  24884,
  26215,
  27103,
  27205,
  31644,
  31854,
  33014,
  37136,
  37139,
  37323,
  37867,
  40794,
  40950,
  41006,
  41247,
  42788,
  42830,
  44018,
  44342,
  44343,
  44629,
  47275,
  48209,
  48439,
  49238,
  57600,
  59914,
  60061,
  66838,
  66908,
  69807,
  69967,
  70157,
  70826,
  73483],
 'PR-агентство': [14127],
 'Ёлки и ёлочные игрушки': [27476,
  37332,
  45465,
  45868,
  55697,
  60437,
  68783,
  76733],
 'Ёлочный базар': [45868, 61569],
 'Ёмкостное оборудование, резервуары': [16960, 567

In [6]:
# Группируем рубрики для OR-логики
rubric_groups = {i: group for i, group in enumerate(rubrics)}
rubric_groups

{0: ['Ресторан', 'Кафе', 'Место для пикника'],
 1: ['Суши-бар', 'Доставка еды и обедов'],
 2: ['Автомойка', 'Детейлинг'],
 3: ['Православный храм']}

In [7]:
def count_rubrics_nearby(dist_matrix, R, rubric_groups):
    n_points = dist_matrix.shape[0]
    results = {}
    
    for group_idx, group_rubrics in rubric_groups.items():
        # Индексы точек df_base, содержащих хотя бы одну рубрику из группы
        matching_indices = set()
        for rubric in group_rubrics:
            matching_indices.update(rubric_to_indices.get(rubric, []))
        matching_indices = np.array(list(matching_indices))
        
        if len(matching_indices) == 0:
            continue
            
        # Для каждой точки из df_points: кол-во соседей в R
        nearby_counts = np.sum(dist_matrix[:, matching_indices] <= R, axis=1)
        
        group_name = '_'.join(group_rubrics)
        results[group_name] = nearby_counts
    
    # Возвращаем DataFrame с колонками по группам
    result_df = pd.DataFrame(results, index=df_points.index)
    return result_df

In [8]:
ls = haversine_2D_mat(df_points, df_base)

count_rubrics_nearby(ls, R, rubric_groups)

,Ресторан_Кафе_Место для пикника,Суши-бар_Доставка еды и обедов,Автомойка_Детейлинг,Православный храм
0,0,0,0,0
1,8,2,1,0
2,8,2,1,0
3,68,2,1,1
4,10,1,0,0
...,...,...,...,...
995,0,0,0,0
996,9,0,1,0
997,0,0,0,0
998,13,4,1,1


In [ ]:
np.argwhere(ls <= R)

In [ ]:
r = df_base.loc[ls <= R, "rubrics"]

In [ ]:

    r = df.loc[ls <= R, "rubrics"]
    rub_split = r.str.split(";").dropna()
    rub_split = rub_split.apply(lambda x: set(x))

In [ ]:
R = 1000
rub_lst = get_list(55.810501, 37.518897, R, [["Ресторан", "Кафе"], ["Кафе"], ["Ресторан"]])
rub_lst

In [ ]:
M = haversine_2D_mat(point_df, df)

R = 500
point_df.loc[M <= R]

# Hide

In [15]:
from collections import Counter

def get_list(lat, lon, R, rub):
    point_df = pd.DataFrame({"lat": [lat], "lon": [lon]})
    ls = haversine_2D_mat(point_df, df_base)[0]
    r = df_base.loc[ls <= R, "rubrics"]
    rub_split = r.str.split(";").dropna()
    rub_split = rub_split.apply(lambda x: set(x))
    lst = {}
    for sub in rub_split:
        for a in rub:
            fl = False
            for x in a:
                if x in sub:
                    fl = True
            if fl:
                lst["_".join(a)] = lst.get("_".join(a), 0) + 1
    return lst

In [16]:
lat = df_points.iloc[1]["lat"]
lon = df_points.iloc[1]["lon"]
get_list(lat, lon, R, rubrics)

{'Ресторан_Кафе_Место для пикника': 8,
 'Суши-бар_Доставка еды и обедов': 2,
 'Автомойка_Детейлинг': 1}

In [ ]:
import time

def f(x):
    time.sleep(x)
    return 2 * x + 1

def g(y):
    time.sleep(y)
    return 3 * y ** 2 - 2

In [ ]:
%%time
f_val = f(12)
g_val = g(8)
f_val + g_val

In [ ]:
import asyncio

async def f(x):
    await asyncio.sleep(x)
    return 2 * x + 1

async def g(y):
    await asyncio.sleep(y)
    return 3 * y ** 2 - 2

In [ ]:
start = time.time()
f_val, g_val = await asyncio.gather(f(12), g(8))
print(f_val + g_val)
print(time.time() - start)

In [ ]:
async def API(lat, lon, R):
    ...

async def yandex_base(lat, lon, R, rubricks):
    ...

api_data, ya_data = await asyncio.gather(API(...), yandex_base(...))
df = pd.concat([api_data, ya_data])